# Ticket Analytics — Module 3: Engineer Suggestion & Assignment
## AI-Driven Intelligent IT Operations Platform | Innodatatics Capstone — ISB AMPBA 2025W

This notebook is **Module 3 of 4** in the ticket analytics stream.

### Depends on
Run **NB1 then NB2** first. This notebook loads all pkl outputs from both.

### What this notebook does
- Loads the **2025 ENGINEER_MASTER** (only) for engineer profiles
- Derives AHP-based assignment weights (Saaty, 1977) — scientifically grounded, CR < 0.10 for all tiers
- Integrates SLA breach scores from the SLA Breach notebook (if available)
- Scores and ranks engineers for each open ticket using a 5-component composite score with tier-adaptive weights
- Generates dynamic, scenario-aware remarks for every assignment action
- Logs every action with actor, timestamp, and detail
- Writes enriched data back to `ticket_assignment_synthetic.xlsx` (5 sheets including new Engineer_Suggestions and Action_Log)
- Exports `tickets_enriched.json` and `ticket_volume_trend.json` for the dashboard

### Outputs
| File | Type | Used by |
|---|---|---|
| `engineer_availability.pkl` | pkl | NB4 |
| `engineer_suggestion_table.pkl` | pkl | NB4 |
| `action_log.pkl` | pkl | NB4 |
| `scoring_output.pkl` | pkl | NB4 |
| `tickets_enriched.json` | json | Dashboard |
| `ticket_volume_trend.json` | json | Dashboard |
| `ticket_assignment_synthetic.xlsx` | xlsx | Reporting |


In [1]:
# ============================================================
# COLAB + GOOGLE DRIVE SETUP — run this cell first every session
# ============================================================
# This cell mounts Google Drive and sets all path variables so every
# notebook reads data from Drive and writes outputs back to Drive.
# No manual uploads or downloads are needed between sessions.
#
# One-time folder structure in your Drive:
#   My Drive/
#   └── Innodatatics_Capstone/
#       ├── data/                     ← all Excel files go here
#       ├── NLP_Ticket_Analytics/     ← all 4 notebooks + schema_utils.py
#       ├── ticket_pkl_outputs/       ← auto-created
#       ├── ticket_eda_plots/         ← auto-created
#       ├── ticket_dashboard_outputs/ ← auto-created
#       └── model_outputs/            ← auto-created
# ============================================================

from google.colab import drive
import os
import sys

drive.mount("/content/drive", force_remount=False)

# Change BASE if your Drive path is different
BASE = "/content/drive/MyDrive/Innodatatics_Capstone"
NLP  = f"{BASE}/NLP_Ticket_Analytics"
DATA = f"{BASE}/data"

os.chdir(NLP)
sys.path.insert(0, NLP)

from pathlib import Path

PKL_DIR  = Path(f"{BASE}/ticket_pkl_outputs");      PKL_DIR.mkdir(exist_ok=True)
PLOT_DIR = Path(f"{BASE}/ticket_eda_plots");         PLOT_DIR.mkdir(exist_ok=True)
DASH_DIR = Path(f"{BASE}/ticket_dashboard_outputs"); DASH_DIR.mkdir(exist_ok=True)
JSON_DIR = Path(f"{BASE}/model_outputs");            JSON_DIR.mkdir(exist_ok=True)

FILES = {
    2023: Path(f"{DATA}/IT_Ops_Intern_Ready_2023.xlsx"),
    2024: Path(f"{DATA}/IT_Ops_Intern_Ready_2024.xlsx"),
    2025: Path(f"{DATA}/IT_Ops_Intern_Ready_2025.xlsx"),
}
SYNTH_FILE      = Path(f"{DATA}/ticket_assignment_synthetic.xlsx")
HIST_SYNTH_FILE = Path(f"{DATA}/ticket_assignment_history_synthetic.xlsx")
ENG_FILE_2025   = Path(f"{DATA}/IT_Ops_Intern_Ready_2025.xlsx")

print("✅  Drive mounted")
print(f"   Working dir : {os.getcwd()}")
print(f"   PKL dir     : {PKL_DIR}")
print(f"   Data files  : {[f.name for f in FILES.values()]}")
missing = [str(p) for p in FILES.values() if not p.exists()]
if missing:
    print(f"\n⚠️  Missing: {missing}")
    print("   Upload these to your Drive data/ folder and re-run.")
else:
    print("   All data files found ✅")


Mounted at /content/drive
✅  Drive mounted
   Working dir : /content/drive/MyDrive/Innodatatics_Capstone/NLP_Ticket_Analytics
   PKL dir     : /content/drive/MyDrive/Innodatatics_Capstone/ticket_pkl_outputs
   Data files  : ['IT_Ops_Intern_Ready_2023.xlsx', 'IT_Ops_Intern_Ready_2024.xlsx', 'IT_Ops_Intern_Ready_2025.xlsx']
   All data files found ✅


In [2]:
\
# =========================
# 0A. Google Drive setup
# =========================
# Run this cell first at the start of every Colab session.
# It mounts your Google Drive and sets all paths so every
# notebook reads and writes directly to Drive — no manual
# uploading or downloading between sessions required.
#
# One-time folder structure to create in Drive:
#   My Drive/
#   └── Innodatatics_Capstone/
#       ├── data/              ← put all 5 xlsx files here
#       ├── NLP_Ticket_Analytics/  ← put all 4 ipynb + schema_utils.py here
#       ├── ticket_pkl_outputs/    ← created automatically
#       ├── ticket_eda_plots/      ← created automatically
#       ├── ticket_dashboard_outputs/ ← created automatically
#       └── model_outputs/         ← created automatically

from google.colab import drive
import os, sys

drive.mount("/content/drive", force_remount=False)

BASE = "/content/drive/MyDrive/Innodatatics_Capstone"
NLP  = f"{BASE}/NLP_Ticket_Analytics"
DATA = f"{BASE}/data"

os.chdir(NLP)
sys.path.insert(0, NLP)

from pathlib import Path
PKL_DIR  = Path(f"{BASE}/ticket_pkl_outputs");      PKL_DIR.mkdir(exist_ok=True)
PLOT_DIR = Path(f"{BASE}/ticket_eda_plots");         PLOT_DIR.mkdir(exist_ok=True)
DASH_DIR = Path(f"{BASE}/ticket_dashboard_outputs"); DASH_DIR.mkdir(exist_ok=True)
JSON_DIR = Path(f"{BASE}/model_outputs");            JSON_DIR.mkdir(exist_ok=True)

FILES = {
    2023: Path(f"{DATA}/IT_Ops_Intern_Ready_2023.xlsx"),
    2024: Path(f"{DATA}/IT_Ops_Intern_Ready_2024.xlsx"),
    2025: Path(f"{DATA}/IT_Ops_Intern_Ready_2025.xlsx"),
}
SYNTH_FILE      = Path(f"{DATA}/ticket_assignment_synthetic.xlsx")
HIST_SYNTH_FILE = Path(f"{DATA}/ticket_assignment_history_synthetic.xlsx")
ENG_FILE_2025   = Path(f"{DATA}/IT_Ops_Intern_Ready_2025.xlsx")

print("✅  Drive mounted")
print(f"   Working dir : {os.getcwd()}")
print(f"   PKL dir     : {PKL_DIR}")
print(f"   Data files  : {[f.name for f in FILES.values()]}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅  Drive mounted
   Working dir : /content/drive/MyDrive/Innodatatics_Capstone/NLP_Ticket_Analytics
   PKL dir     : /content/drive/MyDrive/Innodatatics_Capstone/ticket_pkl_outputs
   Data files  : ['IT_Ops_Intern_Ready_2023.xlsx', 'IT_Ops_Intern_Ready_2024.xlsx', 'IT_Ops_Intern_Ready_2025.xlsx']


In [4]:
# =========================
# 1A. Install libraries if packages are not already included
# =========================
# %pip install pandas numpy scikit-learn openpyxl pathlib pickle5


  Using cached pathlib-1.0.1-py3-none-any.whl.metadata (5.1 kB)
  Using cached pickle5-0.0.11.tar.gz (132 kB)
  Preparing metadata (setup.py) ... done
Using cached pathlib-1.0.1-py3-none-any.whl (14 kB)
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for pickle5
  Running setup.py clean for pickle5
Failed to build pickle5
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (pickle5)


In [5]:
# =========================
# Pickle compatibility — define before loading any pkl from NB2
# =========================
# trained_models.pkl was saved in NB2 with a pipeline that contains
# flatten_text_column inside a FunctionTransformer.
# Pickle stores it by reference to its name in __main__.
# NB3 must define the same function with the same name BEFORE loading
# the pkl, otherwise Python can't find it and raises AttributeError.

def flatten_text_column(x):
    """
    Flatten a text column to a plain string Series.
    Defined here for pickle compatibility — must match the version in NB2 exactly.
    """
    import pandas as pd
    if hasattr(x, "fillna"):
        return x.fillna("unknown").astype(str)
    return pd.Series(x).fillna("unknown").astype(str)

print("✅  flatten_text_column registered for pickle compatibility")

✅  flatten_text_column registered for pickle compatibility


In [6]:
# =========================
# 1B. Imports and load data
# =========================
import warnings
warnings.filterwarnings("ignore")

import pickle, os, json
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

RANDOM_STATE = 42
SYSTEM_USER  = "ai_engine_v1"
LOG_TS       = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
OUT_DIR      = Path(str(PKL_DIR).replace("ticket_pkl_outputs","ticket_assignment_outputs"))
OUT_DIR.mkdir(exist_ok=True)

def load_pkl(name):
    path = PKL_DIR / name
    if not path.exists(): print(f"⚠️  {name} not found"); return None
    with open(path,"rb") as f: return pickle.load(f)
def save_pkl(obj,name):
    with open(PKL_DIR/name,"wb") as f: pickle.dump(obj,f)
    print(f"  ✅  Saved → {PKL_DIR/name}")

base            = load_pkl("ticket_base.pkl")
trained_models  = load_pkl("trained_models.pkl")
best_models     = load_pkl("best_models.pkl")
label_encoders  = load_pkl("label_encoders.pkl")
reco_vectorizer = load_pkl("reco_vectorizer.pkl")
reco_matrix     = load_pkl("reco_matrix.pkl")
reco_base       = load_pkl("reco_base.pkl")
open_preds      = load_pkl("open_ticket_predictions.pkl")

print("✅  All module outputs loaded")


✅  All module outputs loaded


# 2. PERMANENT REFERENCE TABLES

## Objective
Load the three permanent reference sheets from `ticket_assignment_synthetic.xlsx`.
These tables are treated as the source of truth throughout the assignment pipeline.

## Sheets
- **Scenarios** — 20 test scenarios with descriptions and objectives.
  Used as the remark template lookup. Never modified.
- **ticket_assignment_history** — assignment records for all synthetic tickets.
  New columns are appended here by the write-back step.
- **engineer_availability** — engineer profiles and current load.
  Overwritten with live availability at end of each run.


In [7]:
# =========================
# 2A. Load all reference sheets
# =========================
assert SYNTH_FILE.exists(), f"Missing: {SYNTH_FILE}"

# Permanent scenario reference table — never modified
scenarios_df = pd.read_excel(SYNTH_FILE, sheet_name="Scenarios")
print(f"✅  Scenarios table: {scenarios_df.shape}")
display(scenarios_df)

# Engineer availability snapshot
eng_avail_df = pd.read_excel(SYNTH_FILE, sheet_name="engineer_availability")
print(f"\n✅  engineer_availability: {eng_avail_df.shape}")
display(eng_avail_df)

# Assignment history
synth_ah = pd.read_excel(SYNTH_FILE, sheet_name="ticket_assignment_history")
for col in ["assigned_at","unassigned_at","resolved_at"]:
    synth_ah[col] = pd.to_datetime(synth_ah[col], errors="coerce")
print(f"\n✅  ticket_assignment_history: {synth_ah.shape}")

open_tickets = synth_ah[
    (synth_ah["is_current"]==True) &
    (synth_ah["ticket_status"].str.upper().ne("RESOLVED")) &
    (synth_ah["resolved_at"].isna())
].copy()
print(f"   Open tickets for assignment processing: {len(open_tickets)}")


✅  Scenarios table: (20, 3)


,Scenario ID,Description,Test Objective
0,S01,"Direct assignment, P1 resolved within SLA — id...",Verify direct assignment to available engineer...
1,S02,Reassignment due to engineer overload — too ma...,Verify system detects overload and triggers re...
2,S03,"Escalation: junior engineer can't resolve, esc...",Verify escalation logic promotes ticket to sen...
3,S04,Auto round-robin assignment — next available s...,Verify round-robin distributes evenly across a...
4,S05,Skill-mismatch correction — wrong specialist a...,Verify system detects specialization mismatch ...
5,S06,"Engineer unavailable (on leave) — ticket held,...",Verify unavailability flag prevents auto-assig...
6,S07,Multi-ticket load test — same engineer handlin...,Verify system flags engineer at max capacity (...
7,S08,P1 critical — immediate assignment + fast reso...,Verify P1 tickets are immediately routed to hi...
8,S09,"P4 low priority — delayed assignment, slow queue",Verify P4 tickets enter lower-priority queue; ...
9,S10,Cross-specialization: Server engineer covers N...,Verify cross-specialization fallback triggers ...



✅  engineer_availability: (20, 10)


,engineer_id,engineer_name,specialization,support_level,experience_years,shift_type,current_active,bandwidth_status,sla_success_rate,updated_at
0,E001,Eng_1,Network,L2,6,Day,2,Moderate Load,0.592593,2026-05-20 04:06:42
1,E002,Eng_2,Network,L2,7,Day,1,Available,0.593110,2026-05-20 04:06:42
2,E003,Eng_3,Network,L2,1,Day,0,Available,0.619565,2026-05-20 04:06:42
3,E004,Eng_4,Server,L2,6,Day,0,Available,0.611358,2026-05-20 04:06:42
4,E005,Eng_5,Network,L2,6,Day,0,Available,0.606355,2026-05-20 04:06:42
5,E006,Eng_6,Network,L2,4,Day,0,Available,0.607442,2026-05-20 04:06:42
6,E007,Eng_7,Network,L2,6,Day,1,Available,0.623037,2026-05-20 04:06:42
7,E008,Eng_8,Server,L2,10,Day,2,Moderate Load,0.624007,2026-05-20 04:06:42
8,E009,Eng_9,Network,L2,4,Day,0,Available,0.612478,2026-05-20 04:06:42
9,E010,Eng_10,Network,L2,3,Day,1,Available,0.592241,2026-05-20 04:06:42



✅  ticket_assignment_history: (33, 33)
   Open tickets for assignment processing: 16


# 3. AHP-DERIVED WEIGHTS

## Objective
Derive engineer scoring weights scientifically using the
Analytic Hierarchy Process (Saaty, 1977).

## Why AHP instead of regression?
All three data-driven methods tested on the 3-year ticket dataset
returned near-zero signal (AUC=0.507, r<0.04) because the synthetic
breach flag is uniformly distributed at ~40% across all engineer attributes.

AHP is used when domain expertise must substitute for data signal.
The pairwise comparison matrix makes the reasoning explicit and the
Consistency Ratio (CR) validates internal coherence — CR < 0.10 required.

## Weight interpretation by breach tier

| Feature | Low (prob<0.40) | Medium (0.40-0.70) | High (prob≥0.70) |
|---|---|---|---|
| Spec Match | 0.307 | 0.215 | 0.156 |
| Exp Score | 0.140 | 0.174 | 0.156 |
| SLA Success Rate | **0.222** | **0.314** | **0.451** |
| Load Score | 0.241 | 0.198 | 0.156 |
| Issue Familiarity | 0.090 | 0.099 | 0.081 |

At High breach risk, SLA success rate dominates at **45.1%** because
the hard experience filter (≥5yr) already handles quality floor —
breach avoidance history becomes the true differentiator.


In [8]:
# =========================
# 3A. AHP weight derivation
# =========================
import numpy as np

FEATURE_NAMES  = ["spec_match","exp_score","sla_success_rate","load_score","issue_score"]
FEATURE_LABELS = ["Spec Match","Exp Score","SLA Success Rate","Load Score","Issue Familiarity"]

def ahp_weights(matrix, label, print_output=True):
    """
    Derive weights from a pairwise comparison matrix using AHP.
    A[i,j] = how much more important feature i is than feature j.
    Scale: 1=equal, 3=moderate, 5=strong, 7=very strong, 9=absolute.
    Returns (weight_dict, consistency_ratio).
    CR < 0.10 confirms judgments are internally consistent.
    """
    A = np.array(matrix, dtype=float)
    n = A.shape[0]
    A_norm  = A / A.sum(axis=0)
    weights = A_norm.mean(axis=1)
    lam_max = (A @ weights / weights).mean()
    CI  = (lam_max - n) / (n - 1)
    RI  = {1:0, 2:0, 3:0.58, 4:0.90, 5:1.12}
    CR  = CI / RI[n]
    if print_output:
        print(f"\n{'─'*55}")
        print(f"AHP Weights — {label}  (CR={CR:.3f}, {'✅ consistent' if CR<0.10 else '⚠️ INCONSISTENT'})")
        for fn, w in zip(FEATURE_LABELS, weights):
            bar = '█' * int(w * 40)
            print(f"  {fn:22s}: {w:.4f} ({w*100:4.1f}%)  {bar}")
    return dict(zip(FEATURE_NAMES, weights.round(4))), CR

# Low breach risk — specialization match + bandwidth dominate
low_matrix = [
    [1,    2,    1.5,  1.5,  3  ],
    [1/2,  1,    1/2,  1/2,  2  ],
    [1/1.5,2,    1,    1,    2  ],
    [1/1.5,2,    1,    1,    3  ],
    [1/3,  1/2,  1/2,  1/3,  1  ],
]
W_LOW, CR_LOW = ahp_weights(low_matrix, "LOW BREACH RISK (prob < 0.40)")

# Medium breach risk — SLA success rate rises to top
medium_matrix = [
    [1,    1.5,  1/1.5,1,    2  ],
    [1/1.5,1,    1/2,  1,    2  ],
    [1.5,  2,    1,    1.5,  3  ],
    [1,    1,    1/1.5,1,    2  ],
    [1/2,  1/2,  1/3,  1/2,  1  ],
]
W_MED, CR_MED = ahp_weights(medium_matrix, "MEDIUM BREACH RISK (0.40–0.70)")

# High breach risk — SLA success rate dominates at ~45%
# Experience is hard-filtered (≥5yr) so its marginal weight equalizes
high_matrix = [
    [1,    1,    1/3,  1,    2  ],
    [1,    1,    1/3,  1,    2  ],
    [3,    3,    1,    3,    5  ],
    [1,    1,    1/3,  1,    2  ],
    [1/2,  1/2,  1/5,  1/2,  1  ],
]
W_HIGH, CR_HIGH = ahp_weights(high_matrix, "HIGH BREACH RISK (prob ≥ 0.70)")

print(f"\n✅  AHP consistency: Low CR={CR_LOW:.3f} | Med CR={CR_MED:.3f} | High CR={CR_HIGH:.3f}")
print("   All CR < 0.10 — judgments are internally consistent.")

BREACH_HIGH, BREACH_MED = 0.70, 0.40
def breach_tier(p):
    return "High" if p>=BREACH_HIGH else "Medium" if p>=BREACH_MED else "Low"



───────────────────────────────────────────────────────
AHP Weights — LOW BREACH RISK (prob < 0.40)  (CR=0.012, ✅ consistent)
  Spec Match            : 0.3069 (30.7%)  ████████████
  Exp Score             : 0.1399 (14.0%)  █████
  SLA Success Rate      : 0.2224 (22.2%)  ████████
  Load Score            : 0.2406 (24.1%)  █████████
  Issue Familiarity     : 0.0902 ( 9.0%)  ███

───────────────────────────────────────────────────────
AHP Weights — MEDIUM BREACH RISK (0.40–0.70)  (CR=0.005, ✅ consistent)
  Spec Match            : 0.2150 (21.5%)  ████████
  Exp Score             : 0.1735 (17.3%)  ██████
  SLA Success Rate      : 0.3141 (31.4%)  ████████████
  Load Score            : 0.1983 (19.8%)  ███████
  Issue Familiarity     : 0.0991 ( 9.9%)  ███

───────────────────────────────────────────────────────
AHP Weights — HIGH BREACH RISK (prob ≥ 0.70)  (CR=0.001, ✅ consistent)
  Spec Match            : 0.1559 (15.6%)  ██████
  Exp Score             : 0.1559 (15.6%)  ██████
  SLA Success Ra

# 4. ENGINEER PROFILE — 2025 MASTER ONLY

## Objective
Build the engineer performance profile using:
- **Profile** (exp, specialization, shift, support level): 2025 ENGINEER_MASTER exclusively.
- **Historical performance** (SLA success rate, issue counts): all 3 years of ticket history.

## Why 2025 only for profile?
The 2025 ENGINEER_MASTER is the authoritative current state.
Using older years could include engineers who left, changed specialization,
or had different experience levels. Profile must reflect today's reality.
Historical performance uses all 3 years to maximize statistical reliability.


In [9]:
# =========================
# 4A. Load 2025 engineer master + compute historical performance
# =========================
engineers_2025 = pd.read_excel(ENG_FILE_2025, sheet_name="ENGINEER_MASTER")
engineers_2025.columns = [c.strip().lower() for c in engineers_2025.columns]
print(f"✅  2025 ENGINEER_MASTER: {engineers_2025.shape}")
display(engineers_2025)

# Historical performance from all 3 years of ticket data
base_perf = base.copy()
base_perf["sla_num"] = pd.to_numeric(base_perf["sla_breach_flag"],errors="coerce").fillna(0)

eng_hist = (base_perf.groupby("engineer_id")
    .agg(total_handled=("year_ticket_id","count"),
         avg_res_time=("resolution_time_minutes","mean"),
         sla_success_rate=("sla_num",lambda s: 1-s.mean()),
         reopen_rate=("ticket_reopen_flag",lambda s: pd.to_numeric(s,errors="coerce").fillna(0).mean()),
         category_counts=("ticket_category",lambda s: s.value_counts().to_dict()),
         issue_counts=("issue_type",lambda s: s.value_counts().to_dict()))
    .reset_index())

# Merge 2025 profile + historical stats
eng_perf = engineers_2025.merge(eng_hist, on="engineer_id", how="left")
eng_perf["sla_success_rate"] = eng_perf["sla_success_rate"].fillna(0.5)
eng_perf["avg_res_time"]     = eng_perf["avg_res_time"].fillna(eng_perf["avg_res_time"].median())

# Active load from assignment history
active_load = (synth_ah[(synth_ah["is_current"]==True) & (synth_ah["resolved_at"].isna())]
    .groupby("employee_id")
    .agg(current_active=("ticket_id","count"),active_priorities=("ticket_priority",list))
    .reset_index().rename(columns={"employee_id":"engineer_id"}))

eng_perf = eng_perf.merge(active_load, on="engineer_id", how="left")
eng_perf["current_active"] = eng_perf["current_active"].fillna(0).astype(int)

MAX_ACTIVE = 3
eng_perf["bandwidth_status"] = eng_perf["current_active"].apply(
    lambda n: "Available" if n<=1 else "Moderate Load" if n<=2 else "At Capacity")
eng_perf["has_bandwidth"] = eng_perf["current_active"] < MAX_ACTIVE

print("\nEngineer profile (2025 master + historical performance):")
display(eng_perf[["engineer_id","engineer_name","specialization","experience_years",
                   "shift_type","support_level","current_active","bandwidth_status",
                   "sla_success_rate","avg_res_time"]])


✅  2025 ENGINEER_MASTER: (20, 7)


,engineer_id,engineer_name,specialization,support_level,experience_years,shift_type,active_year
0,E001,Eng_1,Network,L2,6,Day,2025
1,E002,Eng_2,Network,L2,7,Day,2025
2,E003,Eng_3,Network,L2,1,Day,2025
3,E004,Eng_4,Server,L2,6,Day,2025
4,E005,Eng_5,Network,L2,6,Day,2025
5,E006,Eng_6,Network,L2,4,Day,2025
6,E007,Eng_7,Network,L2,6,Day,2025
7,E008,Eng_8,Server,L2,10,Day,2025
8,E009,Eng_9,Network,L2,4,Day,2025
9,E010,Eng_10,Network,L2,3,Day,2025



Engineer profile (2025 master + historical performance):


,engineer_id,engineer_name,specialization,experience_years,shift_type,support_level,current_active,bandwidth_status,sla_success_rate,avg_res_time
0,E001,Eng_1,Network,6,Day,L2,2,Moderate Load,0.592593,1583.104139
1,E002,Eng_2,Network,7,Day,L2,1,Available,0.593110,1568.170932
2,E003,Eng_3,Network,1,Day,L2,0,Available,0.619565,1637.132201
3,E004,Eng_4,Server,6,Day,L2,0,Available,0.611358,1576.068341
4,E005,Eng_5,Network,6,Day,L2,0,Available,0.606355,1648.255751
5,E006,Eng_6,Network,4,Day,L2,0,Available,0.607442,1579.203237
6,E007,Eng_7,Network,6,Day,L2,1,Available,0.623037,1676.394606
7,E008,Eng_8,Server,10,Day,L2,2,Moderate Load,0.624007,1648.711703
8,E009,Eng_9,Network,4,Day,L2,0,Available,0.612478,1537.230255
9,E010,Eng_10,Network,3,Day,L2,1,Available,0.592241,1664.726273


In [10]:
# =========================
# 4B. SLA breach score integration
# =========================
# If sla_breach_scores.pkl exists (from the SLA Breach notebook), load it.
# When breach probability is high, suggest_engineers() will:
#   - filter to engineers with experience_years >= 5
#   - shift scoring weight toward sla_success_rate (0.35 → 0.451)
#   - flag ticket for manager review
# If not available, breach_prob defaults to 0.0 (Low tier — standard weights).

SLA_PKL = PKL_DIR / "sla_breach_scores.pkl"
if SLA_PKL.exists():
    sla_scores = load_pkl("sla_breach_scores.pkl")
    SLA_AVAIL  = True
    print(f"✅  SLA breach scores loaded: {sla_scores.shape}")
else:
    sla_scores = None; SLA_AVAIL = False
    print("⚠️  sla_breach_scores.pkl not found.")
    print("   Run Additional_Models_SLA_Breach_Classification_ISB.ipynb and save:")
    print("   sla_breach_scores[['year_ticket_id','sla_breach_prob','sla_risk_tier']]")
    print("   .to_pickle(PKL_DIR/'sla_breach_scores.pkl')")
    print("   Continuing with breach_prob=0.0 (Low tier) as placeholder.")


✅  SLA breach scores loaded: (22476, 10)


In [11]:
# =========================
# 4C. Engineer suggestion function — AHP weights
# =========================
def suggest_engineers(ticket_category, issue_type, ticket_priority,
                      breach_prob=0.0, exclude_ids=None, top_k=5):
    """
    Rank available engineers using AHP-derived composite score.

    Weight sets by breach tier (all CR < 0.10):
      Low  (prob<0.40): spec=0.307, exp=0.140, sla=0.222, load=0.241, issue=0.090
      Med  (0.40-0.70): spec=0.215, exp=0.174, sla=0.314, load=0.198, issue=0.099
      High (prob>=0.70): spec=0.156, exp=0.156, sla=0.451, load=0.156, issue=0.081

    Hard filters before scoring:
      - has_bandwidth: current_active < 3
      - exclude_ids: engineers to skip (already assigned, on leave)
      - High breach: experience_years >= 5
      - Medium breach: experience_years >= 3

    Engineer profiles: 2025 ENGINEER_MASTER only.
    Historical performance: all 3 years of ticket data.
    """
    exclude_ids = exclude_ids or []
    btier = breach_tier(breach_prob)
    pref_spec = "Server" if ticket_category=="Hardware" else "Network"

    W, exp_min = (W_HIGH,5) if btier=="High" else (W_MED,3) if btier=="Medium" else (W_LOW,0)

    candidates = eng_perf[eng_perf["has_bandwidth"] &
                           ~eng_perf["engineer_id"].isin(exclude_ids)].copy()
    if exp_min > 0:
        filt = candidates[candidates["experience_years"]>=exp_min]
        candidates = filt if not filt.empty else candidates

    if candidates.empty:
        candidates = eng_perf[~eng_perf["engineer_id"].isin(exclude_ids)].copy()

    candidates = candidates.copy()
    candidates["f_spec"]  = (candidates["specialization"]==pref_spec).astype(float)
    max_exp = eng_perf["experience_years"].max() or 1
    candidates["f_exp"]   = candidates["experience_years"]/max_exp
    candidates["f_sla"]   = candidates["sla_success_rate"].fillna(0.5)
    candidates["f_load"]  = (1-(candidates["current_active"]/MAX_ACTIVE)).clip(0,1)

    def issue_fam(d,iss):
        if not isinstance(d,dict): return 0.0
        total=sum(d.values()) or 1; return d.get(iss,0)/total
    candidates["f_issue"] = candidates["issue_counts"].apply(lambda d: issue_fam(d,issue_type))
    max_fam = candidates["f_issue"].max() or 1
    candidates["f_issue"] = candidates["f_issue"]/max_fam

    candidates["assignment_score"] = (
        candidates["f_spec"]*W["spec_match"]      +
        candidates["f_exp"] *W["exp_score"]        +
        candidates["f_sla"] *W["sla_success_rate"] +
        candidates["f_load"]*W["load_score"]       +
        candidates["f_issue"]*W["issue_score"]
    ).round(4)

    candidates["breach_risk_tier"] = btier
    candidates["manager_review"]   = btier=="High"
    candidates["score_breakdown"]  = candidates.apply(lambda r:
        f"spec={r['f_spec']*W['spec_match']:.3f}|exp={r['f_exp']*W['exp_score']:.3f}|"
        f"sla={r['f_sla']*W['sla_success_rate']:.3f}|load={r['f_load']*W['load_score']:.3f}|"
        f"issue={r['f_issue']*W['issue_score']:.3f}", axis=1)

    out = ["engineer_id","engineer_name","specialization","experience_years","shift_type",
           "support_level","current_active","bandwidth_status","sla_success_rate","avg_res_time",
           "f_spec","f_exp","f_sla","f_load","f_issue",
           "assignment_score","score_breakdown","breach_risk_tier","manager_review"]
    return (candidates.sort_values("assignment_score",ascending=False)
            [[c for c in out if c in candidates.columns]].head(top_k).reset_index(drop=True))


# 5. ACTION LOGGER

## Objective
Record every assignment action with timestamp, actor, and detail.

## Why logging matters
- Provides a full audit trail for the Mid Review and Final Presentation.
- Eldo Paul specifically requested MoM and action tracking after every call.
- The log is written both to the `Action_Log` sheet in the xlsx and to
  `action_log.pkl`/`action_log.csv` in Drive.


In [12]:
# =========================
# 5A. Action log setup
# =========================
action_log = []

def log_action(action_type, ticket_id, actor, detail,
               target_engineer=None, scenario_id=None):
    """Append a timestamped action entry to the in-memory log."""
    entry = {
        "logged_at":       datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "action_type":     action_type,
        "ticket_id":       ticket_id,
        "actor":           actor,
        "target_engineer": target_engineer or "",
        "scenario_id":     scenario_id or "",
        "detail":          detail,
    }
    action_log.append(entry)
    print(f"  [{entry['logged_at']}] {actor} → {action_type} | {ticket_id} | {detail[:70]}")
    return entry

log_action("MODULE_START","*",SYSTEM_USER,
           f"Module 3 started. Open tickets: {len(open_tickets)}")


  [2026-05-24 16:07:18] ai_engine_v1 → MODULE_START | * | Module 3 started. Open tickets: 16


{'logged_at': '2026-05-24 16:07:18',
 'action_type': 'MODULE_START',
 'ticket_id': '*',
 'actor': 'ai_engine_v1',
 'target_engineer': '',
 'scenario_id': '',
 'detail': 'Module 3 started. Open tickets: 16'}

# 6. ASSIGNMENT PIPELINE

## Objective
For every open ticket in `ticket_assignment_history`:
1. Get model predictions (category, priority, issue, resolution time) from Module 2.
2. Get SLA breach probability from SLA breach model (if available).
3. Generate top-3 engineer suggestions using AHP-weighted scoring.
4. Generate a dynamic remark using the scenario template.
5. Log every action.

## Dynamic remarks
Each of the 20 test scenarios has a template that produces a unique,
context-aware remark using the actual engineer name, ticket attributes,
and breach risk tier detected at runtime.


In [13]:
# =========================
# 6A. Scenario remark templates
# =========================
# One lambda per scenario. Called at runtime with actual values.
SCENARIO_REMARK_TEMPLATES = {
    "S01": lambda e,c,p,i,b: f"Direct assignment to {e} confirmed. {c}/{p}/{i}. Breach tier: {b}.",
    "S02": lambda e,c,p,i,b: f"Reassignment — previous engineer overloaded. {e} assigned ({c}/{p}).",
    "S03": lambda e,c,p,i,b: f"Escalation to senior {e} — junior could not resolve {p} {c} {i}.",
    "S04": lambda e,c,p,i,b: f"Round-robin auto-assignment. {e} next in {c} queue rotation.",
    "S05": lambda e,c,p,i,b: f"Skill-mismatch corrected. {e} reassigned — correct spec for {c}.",
    "S06": lambda e,c,p,i,b: f"Engineer unavailable. After leave check, {e} manually assigned.",
    "S07": lambda e,c,p,i,b: f"Multi-ticket load detected. {e} at capacity — manager flagged.",
    "S08": lambda e,c,p,i,b: f"P1 critical. {e} auto-selected — highest experience for {c} {i}.",
    "S09": lambda e,c,p,i,b: f"P4 low priority — queued. {e} assigned in afternoon slot.",
    "S10": lambda e,c,p,i,b: f"Cross-spec cover. {e} (Server) covering {c} queue — pool full.",
    "S11": lambda e,c,p,i,b: f"SLA breach risk {b}. {e} assigned. Breach monitoring active.",
    "S12": lambda e,c,p,i,b: f"Ticket reopened. {e} reassigned — fresh approach for {c} {i}.",
    "S13": lambda e,c,p,i,b: f"Manager override. {e} selected — preferred for {c} {i} tickets.",
    "S14": lambda e,c,p,i,b: f"Fallback — all preferred engineers busy. {e} next available.",
    "S15": lambda e,c,p,i,b: f"High success rate. {e} selected — best historical {c}/{i}.",
    "S16": lambda e,c,p,i,b: f"Co-assignment — complex ticket. {e} as primary/secondary.",
    "S17": lambda e,c,p,i,b: f"After-hours. {e} assigned via on-call rotation.",
    "S18": lambda e,c,p,i,b: f"Multi-bounce. {e} escalated after {p} {c} bounced twice.",
    "S19": lambda e,c,p,i,b: f"Engineer freed — {e} immediately assigned next ticket in queue.",
    "S20": lambda e,c,p,i,b: f"Bandwidth stress — {e} handling 2 simultaneous P1s. Manager flagged.",
}

def generate_remark(row, sugg_eng, b_tier):
    sid = str(row.get("scenario_id","")).strip()
    fn  = SCENARIO_REMARK_TEMPLATES.get(sid)
    if fn:
        try:
            return fn(sugg_eng or "?",str(row.get("ticket_category","?")),
                      str(row.get("ticket_priority","?")),
                      str(row.get("issue_type","?")), b_tier)
        except: pass
    return (f"Assignment processed for {row.get('ticket_category','?')}/"
            f"{row.get('ticket_priority','?')} ticket. Suggested: {sugg_eng}. Breach: {b_tier}.")


In [14]:
# =========================
# 6B. Run full assignment pipeline
# =========================
eng_suggestion_rows = []
new_ah_cols_rows    = []

for _, row in open_tickets.iterrows():
    ticket_id  = row["ticket_id"]
    assign_id  = row["assignment_id"]
    current_e  = row["employee_id"]
    cat   = str(row.get("ticket_category","Network"))
    prio  = str(row.get("ticket_priority","P3"))
    issue = str(row.get("issue_type","Access"))
    sid   = str(row.get("scenario_id",""))

    # SLA breach probability
    breach_prob = 0.0
    if sla_scores is not None and "ticket_id" in sla_scores.columns:
        m = sla_scores[sla_scores["ticket_id"].astype(str)==str(ticket_id)]
        if not m.empty: breach_prob = float(m["sla_breach_prob"].iloc[0])
    b_tier = breach_tier(breach_prob)

    # Suggest engineers (exclude current if reassignment scenario)
    exclude = [current_e] if sid in ["S02","S03","S05","S06","S12","S13","S14","S18"] else []
    suggestions = suggest_engineers(cat,issue,prio,breach_prob,exclude_ids=exclude,top_k=3)

    e1 = suggestions.iloc[0]["engineer_id"] if len(suggestions)>0 else None
    e2 = suggestions.iloc[1]["engineer_id"] if len(suggestions)>1 else None
    e3 = suggestions.iloc[2]["engineer_id"] if len(suggestions)>2 else None
    sc = round(float(suggestions.iloc[0]["assignment_score"]),4) if len(suggestions)>0 else None
    sp = suggestions.iloc[0]["specialization"]   if len(suggestions)>0 else None
    mg = bool(suggestions.iloc[0]["manager_review"]) if len(suggestions)>0 else False
    sb = suggestions.iloc[0]["score_breakdown"] if len(suggestions)>0 else ""

    # Prediction data from Module 2
    pr = open_preds[open_preds["assignment_id"]==assign_id] if open_preds is not None and len(open_preds)>0 else pd.DataFrame()
    pred_cat  = pr["pred_ticket_category"].iloc[0] if len(pr)>0 and "pred_ticket_category" in pr else cat
    pred_prio = pr["pred_ticket_priority"].iloc[0]  if len(pr)>0 and "pred_ticket_priority" in pr else prio
    pred_iss  = pr["pred_issue_type"].iloc[0]       if len(pr)>0 and "pred_issue_type" in pr else issue
    pred_res  = pr["pred_resolution_time"].iloc[0]  if len(pr)>0 and "pred_resolution_time" in pr else None
    sugg_res  = pr["suggested_resolution"].iloc[0]  if len(pr)>0 and "suggested_resolution" in pr else None
    eng_grp   = pr["recommended_eng_group"].iloc[0] if len(pr)>0 and "recommended_eng_group" in pr else sp

    dynamic_remark = generate_remark(row, e1, b_tier)
    at = ("ESCALATION" if sid in ["S03","S18"] else
          "MANUAL"     if sid in ["S02","S05","S06","S12","S13"] else "AUTO")
    actor = SYSTEM_USER if at=="AUTO" else "manager"

    log_action("SUGGESTION_GENERATED", ticket_id, SYSTEM_USER,
               f"Top: {e1} (score={sc}) | breach={b_tier}", target_engineer=e1, scenario_id=sid)
    if mg:
        log_action("MANAGER_REVIEW_FLAG", ticket_id, SYSTEM_USER,
                   f"breach_prob={breach_prob:.2f} >= {BREACH_HIGH}", target_engineer=e1, scenario_id=sid)
    log_action("ASSIGNMENT_PROCESSED", ticket_id, actor,
               dynamic_remark[:100], target_engineer=e1, scenario_id=sid)

    eng_suggestion_rows.append({
        "assignment_id":        assign_id, "ticket_id":ticket_id, "scenario_id":sid,
        "ticket_category":cat,  "ticket_priority":prio, "issue_type":issue,
        "current_engineer":     current_e,
        "suggested_engineer_1": e1, "suggested_engineer_2":e2, "suggested_engineer_3":e3,
        "suggestion_score_1":   sc, "suggested_specialization":sp,
        "score_breakdown":      sb,
        "sla_breach_prob":      round(breach_prob,4), "sla_breach_tier":b_tier,
        "manager_review":       mg, "assignment_type":at, "actor":actor, "processed_at":LOG_TS,
    })
    new_ah_cols_rows.append({
        "assignment_id":               assign_id,
        "pred_ticket_category":        pred_cat, "pred_ticket_priority":pred_prio,
        "pred_issue_type":             pred_iss,
        "pred_resolution_time_minutes":round(float(pred_res),2) if pred_res else None,
        "sla_breach_prob":             round(breach_prob,4), "sla_breach_tier":b_tier,
        "suggested_engineer_1":        e1, "suggested_engineer_2":e2, "suggested_engineer_3":e3,
        "assignment_score":            sc, "suggested_eng_group":eng_grp,
        "suggested_resolution":        str(sugg_res)[:200] if sugg_res else None,
        "dynamic_remarks":             dynamic_remark, "action_type":at,
        "processed_by":                actor, "processed_at":LOG_TS, "manager_review_flag":mg,
    })

eng_suggestion_df = pd.DataFrame(eng_suggestion_rows)
new_ah_cols_df    = pd.DataFrame(new_ah_cols_rows)
print(f"\nSuggestions generated: {len(eng_suggestion_df)}")
display(eng_suggestion_df)
log_action("BATCH_COMPLETE","*",SYSTEM_USER,
           f"Pipeline complete. {len(eng_suggestion_df)} tickets processed.")


  [2026-05-24 16:07:46] ai_engine_v1 → SUGGESTION_GENERATED | T00041002 | Top: E012 (score=0.8468) | breach=Low
  [2026-05-24 16:07:46] manager → ASSIGNMENT_PROCESSED | T00041002 | Reassignment — previous engineer overloaded. E012 assigned (Network/P2
  [2026-05-24 16:07:46] ai_engine_v1 → SUGGESTION_GENERATED | T00041003 | Top: E012 (score=0.8486) | breach=Low
  [2026-05-24 16:07:46] manager → ASSIGNMENT_PROCESSED | T00041003 | Escalation to senior E012 — junior could not resolve P1 Network Crash.
  [2026-05-24 16:07:46] ai_engine_v1 → SUGGESTION_GENERATED | T00041005 | Top: E005 (score=0.8471) | breach=Low
  [2026-05-24 16:07:46] manager → ASSIGNMENT_PROCESSED | T00041005 | Skill-mismatch corrected. E005 reassigned — correct spec for Network.
  [2026-05-24 16:07:46] ai_engine_v1 → SUGGESTION_GENERATED | T00041006 | Top: E012 (score=0.8468) | breach=Low
  [2026-05-24 16:07:46] manager → ASSIGNMENT_PROCESSED | T00041006 | Engineer unavailable. After leave check, E012 manually assigned.

,assignment_id,ticket_id,scenario_id,ticket_category,ticket_priority,issue_type,current_engineer,suggested_engineer_1,suggested_engineer_2,suggested_engineer_3,suggestion_score_1,suggested_specialization,score_breakdown,sla_breach_prob,sla_breach_tier,manager_review,assignment_type,actor,processed_at
0,13,T00041002,S02,Network,P2,Slow,E007,E012,E005,E009,0.8468,Network,spec=0.307|exp=0.084|sla=0.133|load=0.241|issu...,0.0,Low,False,MANUAL,manager,2026-05-24 16:06:04
1,15,T00041003,S03,Network,P1,Crash,E015,E012,E005,E009,0.8486,Network,spec=0.307|exp=0.084|sla=0.133|load=0.241|issu...,0.0,Low,False,ESCALATION,manager,2026-05-24 16:06:04
2,18,T00041005,S05,Network,P2,Access,E001,E005,E012,E009,0.8471,Network,spec=0.307|exp=0.084|sla=0.135|load=0.241|issu...,0.0,Low,False,MANUAL,manager,2026-05-24 16:06:04
3,20,T00041006,S06,Network,P3,Slow,E017,E012,E005,E009,0.8468,Network,spec=0.307|exp=0.084|sla=0.133|load=0.241|issu...,0.0,Low,False,MANUAL,manager,2026-05-24 16:06:04
4,21,T00041007,S07,Network,P2,Slow,E016,E012,E005,E009,0.8468,Network,spec=0.307|exp=0.084|sla=0.133|load=0.241|issu...,0.0,Low,False,AUTO,ai_engine_v1,2026-05-24 16:06:04
5,22,T00041008,S07,Network,P3,Access,E016,E005,E012,E009,0.8471,Network,spec=0.307|exp=0.084|sla=0.135|load=0.241|issu...,0.0,Low,False,AUTO,ai_engine_v1,2026-05-24 16:06:04
6,23,T00041009,S07,Network,P3,Failure,E016,E005,E012,E006,0.8565,Network,spec=0.307|exp=0.084|sla=0.135|load=0.241|issu...,0.0,Low,False,AUTO,ai_engine_v1,2026-05-24 16:06:04
7,25,T00041011,S09,Network,P4,Access,E010,E005,E012,E009,0.8471,Network,spec=0.307|exp=0.084|sla=0.135|load=0.241|issu...,0.0,Low,False,AUTO,ai_engine_v1,2026-05-24 16:06:04
8,26,T00041012,S10,Network,P2,Failure,E018,E005,E012,E006,0.8565,Network,spec=0.307|exp=0.084|sla=0.135|load=0.241|issu...,0.0,Low,False,AUTO,ai_engine_v1,2026-05-24 16:06:04
9,29,T00041014,S12,Hardware,P3,Crash,E018,E004,E013,E019,0.8524,Server,spec=0.307|exp=0.084|sla=0.136|load=0.241|issu...,0.0,Low,False,MANUAL,manager,2026-05-24 16:06:04


  [2026-05-24 16:07:47] ai_engine_v1 → BATCH_COMPLETE | * | Pipeline complete. 16 tickets processed.


{'logged_at': '2026-05-24 16:07:47',
 'action_type': 'BATCH_COMPLETE',
 'ticket_id': '*',
 'actor': 'ai_engine_v1',
 'target_engineer': '',
 'scenario_id': '',
 'detail': 'Pipeline complete. 16 tickets processed.'}

# 7. WRITE-BACK TO EXCEL

## Objective
Write all computed outputs back into `ticket_assignment_synthetic.xlsx`
as permanent additions to the workbook — no manual copy-paste required.

## What is updated
- `ticket_assignment_history`: 17 new columns appended (predictions,
  SLA scores, engineer suggestions, dynamic remarks, action metadata).
- `engineer_availability`: overwritten with live current load + AHP stats.
- `Engineer_Suggestions`: new permanent sheet — one row per processed ticket.
- `Action_Log`: new permanent sheet — every logged action.


In [15]:
# =========================
# 7A. Excel write-back helpers
# =========================
def style_header(ws, fill_color="0F2744"):
    hf = PatternFill("solid",fgColor=fill_color)
    for cell in ws[1]:
        cell.fill = hf
        cell.font = Font(bold=True,color="FFFFFF",size=11)
        cell.alignment = Alignment(horizontal="center",vertical="center")

def auto_width(ws, min_w=10, max_w=50):
    for col in ws.columns:
        w = max(len(str(cell.value or "")) for cell in col)+2
        ws.column_dimensions[col[0].column_letter].width = min(max(w,min_w),max_w)

even_fill = PatternFill("solid",fgColor="EEF3FA")
odd_fill  = PatternFill("solid",fgColor="FFFFFF")


In [16]:
# =========================
# 7B. Update all sheets and save
# =========================
wb = openpyxl.load_workbook(SYNTH_FILE)
new_data_lookup = {int(r["assignment_id"]): r for r in new_ah_cols_rows}

# ── ticket_assignment_history — append new columns ────────────────────────
ws_ah = wb["ticket_assignment_history"]
existing_headers = [cell.value for cell in ws_ah[1]]
new_cols = ["pred_ticket_category","pred_ticket_priority","pred_issue_type",
            "pred_resolution_time_minutes","sla_breach_prob","sla_breach_tier",
            "suggested_engineer_1","suggested_engineer_2","suggested_engineer_3",
            "assignment_score","suggested_eng_group","suggested_resolution",
            "dynamic_remarks","action_type","processed_by","processed_at","manager_review_flag"]
for c in new_cols:
    if c not in existing_headers: existing_headers.append(c)
for i,h in enumerate(existing_headers,1): ws_ah.cell(1,i).value=h
style_header(ws_ah)
for ri in range(2, ws_ah.max_row+1):
    try: aid=int(ws_ah.cell(ri,1).value)
    except: continue
    enrichment=new_data_lookup.get(aid,{})
    fill=even_fill if ri%2==0 else odd_fill
    for c,v in enrichment.items():
        if c=="assignment_id": continue
        if c in existing_headers:
            cell=ws_ah.cell(ri,existing_headers.index(c)+1)
            cell.value=v; cell.fill=fill
            cell.alignment=Alignment(wrap_text=True,vertical="top")
auto_width(ws_ah)
print("✅  ticket_assignment_history updated")

# ── engineer_availability — overwrite with live data ─────────────────────
eng_updated = eng_perf[["engineer_id","engineer_name","specialization","support_level",
    "experience_years","shift_type","current_active","bandwidth_status","sla_success_rate"]].copy()
eng_updated["updated_at"]=LOG_TS
ws_eng=wb["engineer_availability"]
ws_eng.delete_rows(1,ws_eng.max_row)
ws_eng.append(list(eng_updated.columns)); style_header(ws_eng)
sc={"Available":"D4EDDA","Moderate Load":"FFF3CD","At Capacity":"F8D7DA"}
for i,rd in enumerate(eng_updated.itertuples(index=False),2):
    vals=list(rd); ws_eng.append(vals)
    bw=vals[list(eng_updated.columns).index("bandwidth_status")]
    fill=PatternFill("solid",fgColor=sc.get(str(bw),"FFFFFF"))
    for cell in ws_eng[i]: cell.fill=fill
auto_width(ws_eng); print("✅  engineer_availability updated")

# ── Engineer_Suggestions — new permanent sheet ────────────────────────────
for sn in ["Engineer_Suggestions","Action_Log"]:
    if sn in wb.sheetnames: del wb[sn]

ws_sugg=wb.create_sheet("Engineer_Suggestions")
ws_sugg.append(list(eng_suggestion_df.columns)); style_header(ws_sugg,fill_color="1A4E79")
for i,rd in enumerate(eng_suggestion_df.itertuples(index=False),2):
    ws_sugg.append(list(rd))
    fill=PatternFill("solid",fgColor="F8D7DA" if list(rd)[-2] else "D4EDDA")
    for cell in ws_sugg[i]: cell.fill=fill
auto_width(ws_sugg); print("✅  Engineer_Suggestions sheet created")

# ── Action_Log — new permanent sheet ─────────────────────────────────────
action_log_df=pd.DataFrame(action_log)
ws_log=wb.create_sheet("Action_Log")
ws_log.append(list(action_log_df.columns)); style_header(ws_log,fill_color="2D6A4F")
for i,rd in enumerate(action_log_df.itertuples(index=False),2):
    ws_log.append(list(rd))
    fill=even_fill if i%2==0 else odd_fill
    for cell in ws_log[i]: cell.fill=fill
auto_width(ws_log); print(f"✅  Action_Log created: {len(action_log_df)} entries")

wb.save(SYNTH_FILE)
print(f"\n✅  {SYNTH_FILE.name} saved. Sheets: {wb.sheetnames}")
log_action("FILE_SAVED","*",SYSTEM_USER,f"{SYNTH_FILE.name} saved with all updates")


✅  ticket_assignment_history updated
✅  engineer_availability updated
✅  Engineer_Suggestions sheet created
✅  Action_Log created: 34 entries

✅  ticket_assignment_synthetic.xlsx saved. Sheets: ['Scenarios', 'ticket_assignment_history', 'engineer_availability', 'Engineer_Suggestions', 'Action_Log']
  [2026-05-24 16:07:58] ai_engine_v1 → FILE_SAVED | * | ticket_assignment_synthetic.xlsx saved with all updates


{'logged_at': '2026-05-24 16:07:58',
 'action_type': 'FILE_SAVED',
 'ticket_id': '*',
 'actor': 'ai_engine_v1',
 'target_engineer': '',
 'scenario_id': '',
 'detail': 'ticket_assignment_synthetic.xlsx saved with all updates'}

# 8. JSON EXPORT FOR DASHBOARD

## Objective
Write `tickets_enriched.json` and supporting JSON files to `model_outputs/`.
These files are the contract between the model layer and the dashboard frontend
as defined in `Data_from_Models.docx`.

## Files written
- `tickets_enriched.json` — all tickets with predictions, SLA risk,
  assignment, and similar ticket recommendations.
- `ticket_volume_trend.json` — daily volume + resolved counts.
- `sla_compliance_trend.json` — monthly breach rate vs 99% target.


In [17]:
# =========================
# 8A. Build tickets_enriched.json
# =========================
# NLP fields: ticket_detailed_description (nlp_text_detailed) used for similarity.
# ticket_description is display metadata only — surfaced as "ticket_description" field.

tickets_enriched = []
for _, ah_row in synth_ah.iterrows():
    tid    = ah_row["ticket_id"]
    cat    = str(ah_row.get("ticket_category","Network"))
    prio   = str(ah_row.get("ticket_priority","P3"))
    issue  = str(ah_row.get("issue_type","Access"))
    status = str(ah_row.get("ticket_status","Open"))

    pr = open_preds[open_preds["ticket_id"]==tid] if (open_preds is not None and "ticket_id" in open_preds.columns) else pd.DataFrame()
    sr = eng_suggestion_df[eng_suggestion_df["ticket_id"]==tid] if len(eng_suggestion_df)>0 else pd.DataFrame()

    pred_cat  = pr["pred_ticket_category"].iloc[0] if len(pr)>0 and "pred_ticket_category" in pr else cat
    pred_prio = pr["pred_ticket_priority"].iloc[0]  if len(pr)>0 and "pred_ticket_priority" in pr else prio
    pred_iss  = pr["pred_issue_type"].iloc[0]       if len(pr)>0 and "pred_issue_type" in pr else issue
    pred_res  = pr["pred_resolution_time"].iloc[0]  if len(pr)>0 and "pred_resolution_time" in pr else None
    conf      = pr["conf_ticket_category"].iloc[0]  if len(pr)>0 and "conf_ticket_category" in pr else 0.0
    sugg_res  = pr["suggested_resolution"].iloc[0]  if len(pr)>0 and "suggested_resolution" in pr else None
    eng_grp   = pr["recommended_eng_group"].iloc[0] if len(pr)>0 and "recommended_eng_group" in pr else None
    sugg_e    = sr["suggested_engineer_1"].iloc[0]  if len(sr)>0 else ah_row.get("employee_id","")
    breach_p  = float(sr["sla_breach_prob"].iloc[0])if len(sr)>0 else 0.0
    breach_t  = str(sr["sla_breach_tier"].iloc[0])  if len(sr)>0 else "Low"
    mgr_rev   = bool(sr["manager_review"].iloc[0])  if len(sr)>0 else False
    sla_band  = "High" if breach_p>=0.70 else "Watch" if breach_p>=0.40 else "OnTrack"

    # Similar tickets via recommendation engine
    base_m = base[(base["ticket_category"]==cat)&(base["issue_type"]==issue)].head(1)
    similar = []
    if not base_m.empty and reco_vectorizer is not None:
        try:
            from sklearn.metrics.pairwise import cosine_similarity as _cs
            nlp_q = str(base_m["nlp_text_detailed"].iloc[0])
            q_vec = reco_vectorizer.transform([f"{nlp_q} | {issue} | {cat}"])
            sims  = _cs(q_vec, reco_matrix).flatten()
            tmp   = reco_base.copy(); tmp["sim"]=sims
            recs  = tmp.sort_values("sim",ascending=False).head(3)
            similar = recs[["year_ticket_id","issue_type","resolution_notes",
                             "resolution_time_minutes","sim"]
                          ].rename(columns={"sim":"similarity_score"}).to_dict(orient="records")
        except: pass

    tickets_enriched.append({
        "year_ticket_id":                  tid,
        "ticket_created_timestamp":        str(ah_row.get("assigned_at","")),
        "ticket_description":              str(ah_row.get("remarks","")),
        "ticket_status":                   status,
        "actual_category":                 cat, "actual_priority":prio, "actual_issue_type":issue,
        "predicted_category":              pred_cat, "predicted_issue_type":pred_iss,
        "predicted_priority":              pred_prio,
        "classification_confidence":       round(float(conf),4) if conf else 0.0,
        "predicted_resolution_time_minutes":round(float(pred_res),2) if pred_res else None,
        "sla_breach_probability":          round(breach_p,4),
        "sla_risk_band":                   sla_band, "sla_breach_tier":breach_t,
        "assigned_engineer":               {"engineer_id":ah_row.get("employee_id","")},
        "suggested_engineer":              sugg_e, "assigned_team":eng_grp,
        "manager_review_flag":             mgr_rev,
        "recommendations":                 {"suggested_resolution":str(sugg_res) if sugg_res else None,
                                            "recommended_engineer_group":eng_grp,
                                            "similar_tickets":similar},
        "scenario_id":                     str(ah_row.get("scenario_id","")),
        "assignment_type":                 str(ah_row.get("assignment_type","AUTO")),
    })

with open(JSON_DIR/"tickets_enriched.json","w") as f:
    json.dump(tickets_enriched,f,indent=2,default=str)
print(f"✅  tickets_enriched.json: {len(tickets_enriched)} records → {JSON_DIR}")

# Volume trend and SLA compliance trend
base_ts = base.dropna(subset=["ticket_created_timestamp"]).copy()
base_ts["day"] = base_ts["ticket_created_timestamp"].dt.day_name()
base_ts["sla_n"] = pd.to_numeric(base_ts["sla_breach_flag"],errors="coerce").fillna(0)
vol = (base_ts.groupby("day").agg(total=("year_ticket_id","count"),
        resolved=("ticket_status",lambda s:(s=="Closed").sum()))
       .reset_index().to_dict(orient="records"))
with open(JSON_DIR/"ticket_volume_trend.json","w") as f: json.dump(vol,f,indent=2)

import calendar
monthly=[]
for m in range(1,13):
    sub=base_ts[base_ts["ticket_created_timestamp"].dt.month==m]
    if sub.empty: continue
    monthly.append({"month":calendar.month_abbr[m],
                    "compliance":round((1-sub["sla_n"].mean())*100,2),"target":99.0})
with open(JSON_DIR/"sla_compliance_trend.json","w") as f: json.dump(monthly,f,indent=2)
print(f"✅  ticket_volume_trend.json and sla_compliance_trend.json written")
log_action("JSON_EXPORT","*",SYSTEM_USER,f"{len(tickets_enriched)} tickets exported to JSON")


✅  tickets_enriched.json: 33 records → /content/drive/MyDrive/Innodatatics_Capstone/model_outputs
✅  ticket_volume_trend.json and sla_compliance_trend.json written
  [2026-05-24 16:08:05] ai_engine_v1 → JSON_EXPORT | * | 33 tickets exported to JSON


{'logged_at': '2026-05-24 16:08:05',
 'action_type': 'JSON_EXPORT',
 'ticket_id': '*',
 'actor': 'ai_engine_v1',
 'target_engineer': '',
 'scenario_id': '',
 'detail': '33 tickets exported to JSON'}

# 9. SAVE OUTPUTS

## Objective
Persist all assignment outputs to Google Drive.


In [18]:
# =========================
# 9A. Save all pkl outputs and CSVs
# =========================
log_action("MODULE_END","*",SYSTEM_USER,
           f"Module 3 complete. Log entries: {len(action_log)}")

action_log_df = pd.DataFrame(action_log)
save_pkl(eng_suggestion_df,  "engineer_suggestion_table.pkl")
save_pkl(action_log_df,      "action_log.pkl")
save_pkl(eng_perf,           "engineer_availability.pkl")

action_log_df.to_csv(OUT_DIR/"action_log.csv",index=False)
eng_suggestion_df.to_csv(OUT_DIR/"engineer_suggestions.csv",index=False)

print(f"\n✅  Module 3 complete.")
print(f"   Action log entries  : {len(action_log_df)}")
print(f"   JSON files          : {[f.name for f in JSON_DIR.iterdir()]}")
print(f"   Updated xlsx        : {SYNTH_FILE.name}")
print("   Next step → open and run Ticket_04_Dashboard_Outputs.ipynb")


  [2026-05-24 16:08:08] ai_engine_v1 → MODULE_END | * | Module 3 complete. Log entries: 36
  ✅  Saved → /content/drive/MyDrive/Innodatatics_Capstone/ticket_pkl_outputs/engineer_suggestion_table.pkl
  ✅  Saved → /content/drive/MyDrive/Innodatatics_Capstone/ticket_pkl_outputs/action_log.pkl
  ✅  Saved → /content/drive/MyDrive/Innodatatics_Capstone/ticket_pkl_outputs/engineer_availability.pkl

✅  Module 3 complete.
   Action log entries  : 37
   JSON files          : ['tickets_enriched.json', 'sla_compliance_trend.json', 'ticket_volume_trend.json']
   Updated xlsx        : ticket_assignment_synthetic.xlsx
   Next step → open and run Ticket_04_Dashboard_Outputs.ipynb
